In [ ]:
!pip install requests pandas matplotlib seaborn scipy biopython python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.4 MB/s eta 0:00:00


## final

In [ ]:
import time
import pandas as pd
from Bio import Entrez

In [ ]:

CANDIDATE_FACTORS = [
    'ergotism', 'butyl benzyl phthalate', 'biosolids', 'alkylphenols',
    'mold spores', 'chlorine dioxide', 'radon in water', 'sulfites'
]

In [ ]:
BASELINE_FACTORS = [
    "lead", "tobacco", "air pollution", "arsenic", "occupational exposure"
]

In [ ]:
ALL_FACTORS = BASELINE_FACTORS + CANDIDATE_FACTORS

In [ ]:

intervals = [
    (1940, 1959),
    (1960, 1979),
    (1980, 1999),
    (2000, 2019),
    (2020, 2026)
]

In [ ]:
def safe_entrez_search(term, retmax=0, retries=5, backoff=2):
    for attempt in range(retries):
        try:
            handle = Entrez.esearch(db="pubmed", term=term, retmax=retmax)
            record = Entrez.read(handle)
            handle.close()
            return record
        except Exception as e:
            if attempt == retries - 1:
                raise
            wait = backoff ** attempt
            print(f"Retrying in {wait}s...")
            time.sleep(wait)

In [ ]:
def get_pubmed_count(factor, start_year, end_year):
    query = f'("{factor}"[Title/Abstract]) AND ({start_year}[PDAT] : {end_year}[PDAT])'
    record = safe_entrez_search(query, retmax=0)
    return int(record["Count"])

In [ ]:

def main():
    results = []

    for factor in ALL_FACTORS:
        print(f"Processing: {factor}")
        row = {"Factor": factor, "Type": "Baseline" if factor in BASELINE_FACTORS else "Candidate"}
        total = 0
        for start, end in intervals:
            try:
                count = get_pubmed_count(factor, start, end)
            except Exception as e:
                print(f"  Error for {factor} {start}-{end}: {e}")
                count = 0
            row[f"{start}-{end}"] = count
            total += count
            time.sleep(0.34)  # NCBI rate limit
        row["Total (1940-2026)"] = total
        results.append(row)

    df = pd.DataFrame(results)

    cols = ["Factor", "Type"] + [f"{s}-{e}" for s, e in intervals] + ["Total (1940-2026)"]
    df = df[cols]


    print(df.to_string(index=False))

    df.to_csv("publications_per_20_years.csv", index=False)
    print("\nTable saved as 'publications_per_20_years.csv'")

    return df

if __name__ == "__main__":
    df_table = main()

Processing: lead
Processing: tobacco
Processing: air pollution
Processing: arsenic
Processing: occupational exposure
Processing: ergotism
Processing: butyl benzyl phthalate
Processing: biosolids
Processing: alkylphenols
Processing: mold spores
Processing: chlorine dioxide
Processing: radon in water
Processing: sulfites

PUBLICATION COUNTS PER 20‑YEAR INTERVAL (1940–2026)
                Factor      Type  1940-1959  1960-1979  1980-1999  2000-2019  2020-2026  Total (1940-2026)
                  lead  Baseline       1866      14242     100134     475281     336770             928293
               tobacco  Baseline       1262       4152      19806      73183      40673             139076
         air pollution  Baseline        909       3536       2915      19001      23487              49848
               arsenic  Baseline       1023       1220       2760      22285      14328              41616
 occupational exposure  Baseline         12        454       5212      11618       6327    